In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.sql('select * from databricksreddy.bronze.products')

In [0]:
df.display()

In [0]:
df = df.drop("_rescued_data")

In [0]:
df.display()

##Functions

In [0]:
df.createOrReplaceTempView('products')

In [0]:
%sql
create or replace function databricksreddy.bronze.discount_func(price double)
  returns double
  LANGUAGE sql
  return price * 0.95

In [0]:
%sql
select product_id, databricksreddy.bronze.discount_func(price) as discounted_price
from products

In [0]:
df = df.withColumn('discounted_price',expr('databricksreddy.bronze.discount_func(price)'))
df.display()

In [0]:
%sql
create or replace function databricksreddy.bronze.upper(brand string)
returns string
language python
as
$$  
    return brand.upper()
$$

In [0]:
%sql
select brand, databricksreddy.bronze.upper(category) as brand_upper
from products

In [0]:
df.display()

In [0]:
df.write.format('delta')\
        .mode('overwrite')\
        .save('abfss://silver@datalakevishnu1.dfs.core.windows.net/products')

In [0]:
df.write.mode('overwrite').saveAsTable('databricksreddy.silver.products')

In [0]:
%sql
select * from databricksreddy.silver.products